In [ ]:
!pip install datasets transformers pyarrow pandas Pillow requests -q

# Key Ideas

- State manager used to maintain state as these processes take really long time to run.

- Right now NOT SUPPORTED to run concurrently due to the state management (only accessible directly from 1 instance)

- might need to consider using a more parallelized method to extract images from the URLs and pass it through the CLIPProcessor

### Few key things to check

1. Am I even using CLIPProcessor properly?
2. How can we "split up" the compute appropriatelu
3. Given the existing embeddings right now - is it clear/straight forward how to augment our raw data for augmentation?

In [ ]:
import os
import gc
import json
import time
import logging
import requests
from io import BytesIO
import pandas as pd
import torch
from PIL import Image
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor
from google.colab import drive

# --- Configuration & Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Mount Google Drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Amazon_Reviews_Pipeline'
LOCAL_TMP_DIR = '/content/tmp_pipeline'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(LOCAL_TMP_DIR, exist_ok=True)

STATE_FILE = os.path.join(PROJECT_DIR, 'pipeline_state.json')
FINAL_OUTPUT_DIR = os.path.join(PROJECT_DIR, 'embeddings')
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)

CATEGORIES = ['Clothing_Shoes_and_Jewelry', 'Sports_and_Outdoors', 'Beauty_and_Personal_Care']
START_DATE = pd.to_datetime('2020-01-01', utc=True)
END_DATE = pd.to_datetime('2022-12-31', utc=True)

# --- State Management & 2-Strike DLQ ---
class PipelineState:
    def __init__(self, state_file):
        self.state_file = state_file
        self.state = {'unique_asins': [], 'last_processed_index': 0, 'error_registry': {}}
        self.load_state()

    def load_state(self):
        if os.path.exists(self.state_file):
            try:
                with open(self.state_file, 'r') as f:
                    self.state = json.load(f)
                print(f"Loaded state from {self.state_file}. Resuming from index {self.state['last_processed_index']}")
            except Exception as e:
                print(f"Failed to load state: {e}")

    def save_state(self):
        # Atomic save
        temp_file = self.state_file + '.tmp'
        with open(temp_file, 'w') as f:
            json.dump(self.state, f)
        os.replace(temp_file, self.state_file)

    def register_error(self, asin, error_msg):
        if asin not in self.state['error_registry']:
            self.state['error_registry'][asin] = {'strikes': 1, 'status': 'retrying', 'error': error_msg}
        else:
            self.state['error_registry'][asin]['strikes'] += 1
            if self.state['error_registry'][asin]['strikes'] >= 2:
                self.state['error_registry'][asin]['status'] = 'problematic'
                self.state['error_registry'][asin]['error'] = error_msg
        # Removed self.save_state() here, state will be saved by the main loop.

    def is_problematic(self, asin):
        return self.state['error_registry'].get(asin, {}).get('status') == 'problematic'

# Verify state file location exists and is as expected first before loading state
print(f'State File Location: {STATE_FILE}')

Mounted at /content/drive
State File Location: /content/drive/MyDrive/Amazon_Reviews_Pipeline/pipeline_state.json


In [ ]:
state_manager = PipelineState(STATE_FILE)

Loaded state from /content/drive/MyDrive/Amazon_Reviews_Pipeline/pipeline_state.json. Resuming from index 2216960


In [ ]:
for key in state_manager.state:
    state_value = state_manager.state[key]
    if isinstance(state_value, list):
        print(f"State Manager Key: {key} - Type: list, Length: {len(state_value)}")
        if state_value:
            print("  Sample Values (up to 3):")
            for i, item in enumerate(state_value[:3]):
                print(f"    - {item} (Type: {type(item).__name__})")
    elif isinstance(state_value, dict):
        print(f"State Manager Key: {key} - Type: dict, Length: {len(state_value)}")
        if state_value:
            print("  Sample Keys and their Value Types (up to 3):")
            for i, (subkey, sub_value) in enumerate(list(state_value.items())[:3]):
                print(f"    - Subkey: '{subkey}' - Value Type: {type(sub_value).__name__}")
                if isinstance(sub_value, dict):
                    print("      Nested Dict Keys (up to 3):")
                    for j, (nested_key, nested_value) in enumerate(list(sub_value.items())[:3]):
                        print(f"        - Nested Key: '{nested_key}' - Nested Value Type: {type(nested_value).__name__}")
                        if isinstance(nested_value, int):
                            print(f"          Value: {nested_value}")
                        elif isinstance(nested_value, str):
                            print(f"          Value (first 100 chars): {nested_value[:100]}")
    else:
        print(f"State Manager Key: {key} - Type: {type(state_value).__name__}, Value: {state_value}")

State Manager Key: unique_asins - Type: list, Length: 4381511
  Sample Values (up to 3):
    - B006DEIZT0 (Type: str)
    - B01780QKT4 (Type: str)
    - B07QL1BLHQ (Type: str)
State Manager Key: last_processed_index - Type: int, Value: 2216960
State Manager Key: error_registry - Type: dict, Length: 522
  Sample Keys and their Value Types (up to 3):
    - Subkey: 'B07BYG6WN8' - Value Type: dict
      Nested Dict Keys (up to 3):
        - Nested Key: 'strikes' - Nested Value Type: int
          Value: 1
        - Nested Key: 'status' - Nested Value Type: str
          Value (first 100 chars): retrying
        - Nested Key: 'error' - Nested Value Type: str
          Value (first 100 chars): No image URL found in metadata.
    - Subkey: 'B01IQ5EOAM' - Value Type: dict
      Nested Dict Keys (up to 3):
        - Nested Key: 'strikes' - Nested Value Type: int
          Value: 1
        - Nested Key: 'status' - Nested Value Type: str
          Value (first 100 chars): retrying
        - Neste

In [ ]:
for category in CATEGORIES:
    print(f"Category: {category} - {state_manager.state['processed_categories'].get(f"{category}_last_row", 0)}")

Category: Clothing_Shoes_and_Jewelry - 65000000
Category: Sports_and_Outdoors - 15000000
Category: Beauty_and_Personal_Care - 20000000


# Step 1: Data Filtering & Extraction

In [ ]:
# --- Step 1: Data Filtering & Extraction ---
import os
import json
import requests
from tqdm.auto import tqdm

checkpoints = [10_000, 50_000, 100_000, 500_000, 1_000_000, 5_000_000, 10_000_000, 20_000_000, 30_000_000, 50_000_000]

# Initialize state tracking for categories if not present
if 'processed_categories' not in state_manager.state:
    state_manager.state['processed_categories'] = {}

# Load any previously extracted ASINs from the state
unique_asins = set(state_manager.state.get('unique_asins', []))

def download_with_resume(url, filename):
    """Downloads a file with a tqdm progress bar and supports resuming."""
    file_size = os.path.getsize(filename) if os.path.exists(filename) else 0

    # Get total file size from server
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    if total_size != 0 and file_size >= total_size:
        print(f"File {os.path.basename(filename)} is already fully downloaded.")
        return

    headers = {'Range': f'bytes={file_size}-'}
    response = requests.get(url, headers=headers, stream=True)

    with open(filename, 'ab') as f, tqdm(
        desc=os.path.basename(filename),
        initial=file_size,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=65536):
            size = f.write(data)
            bar.update(size)

for category in CATEGORIES:
    if state_manager.state['processed_categories'].get(category, False):
        print(f"Category {category} already completely processed. Skipping.")
        continue

    print(f"Processing category: {category}")

    # Determine where we left off for this category if we crashed previously
    start_row = state_manager.state['processed_categories'].get(f"{category}_last_row", 0)

    url = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{category}.jsonl"
    local_path = os.path.join(LOCAL_TMP_DIR, f"{category}.jsonl")

    print(f"Downloading {category} to local disk...")
    download_with_resume(url, local_path)

    print(f"Parsing local file {local_path} from row {start_row}...")

    counter = 0
    with open(local_path, 'r', encoding='utf-8') as f:
        for line in f:
            counter += 1
            # Fast forward to where we left off
            if counter <= start_row:
                continue

            if counter in checkpoints or counter % 5_000_000 == 0:
                print(f"Processed {counter} rows so far in {category}...")
                # Save incremental progress
                state_manager.state['unique_asins'] = list(unique_asins)
                state_manager.state['processed_categories'][f"{category}_last_row"] = counter
                state_manager.save_state()

            try:
                row = json.loads(line)
                ts = row.get('timestamp')
                if ts:
                    # Convert timestamp to pandas datetime for easy comparison
                    ts = pd.to_datetime(ts, unit='ms', utc=True) if isinstance(ts, (int, float)) else pd.to_datetime(ts, utc=True)
                    if START_DATE <= ts <= END_DATE:
                        unique_asins.add(row['parent_asin'])
            except Exception as e:
                continue # skip malformed rows

    # Mark category as fully complete
    state_manager.state['processed_categories'][category] = True
    state_manager.state['unique_asins'] = list(unique_asins)
    state_manager.save_state()

    # Clean up local file to save disk space
    if os.path.exists(local_path):
        os.remove(local_path)
        print(f"Deleted local file {local_path}")

print(f"Extraction complete! Found {len(state_manager.state['unique_asins'])} unique ASINs in the 3-year window.")

# Step 2: GPU Pipeline Set-up

In [ ]:
# --- Step 2: GPU Pipeline Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device).eval()

def download_image(url, timeout=5):
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        raise ValueError(f"Image download failed: {str(e)}")

# --- Lookup Dictionary Generation ---
URL_MAPPING_FILE = os.path.join(PROJECT_DIR, 'asin_to_image_url.json')

# Load existing mapping if available
if os.path.exists(URL_MAPPING_FILE):
    with open(URL_MAPPING_FILE, 'r') as f:
        asin_to_image_url = json.load(f)
    print(f"Loaded {len(asin_to_image_url)} ASIN image URLs from {URL_MAPPING_FILE}")
else:
    asin_to_image_url = {}

# Initialize meta tracking if not exists
if 'processed_meta_categories' not in state_manager.state:
    state_manager.state['processed_meta_categories'] = {}

# Initialize and load 'no_image_asins' from state, use a set for efficient lookup during runtime
if 'no_image_asins' not in state_manager.state:
    state_manager.state['no_image_asins'] = []
no_image_asins = set(state_manager.state['no_image_asins'])

target_asins = set(state_manager.state['unique_asins'])

SAVE_FREQUENCY = 50_000
print("Building ASIN to Image URL mapping from meta datasets...")
for category in CATEGORIES:
    if state_manager.state['processed_meta_categories'].get(category):
        print(f"Metadata for {category} already processed. Skipping.")
        continue

    print(f"Processing metadata for {category}...")

    # Stream directly via HTTP to bypass pyarrow strict schema enforcement
    url = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_{category}.jsonl"
    response = requests.get(url, stream=True)
    response.raise_for_status()

    counter = 0
    captured_in_category = 0
    for line in response.iter_lines():
        if not line:
            continue

        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue

        counter += 1
        if counter % SAVE_FREQUENCY == 0:
            print(f"Processed {counter} meta rows in {category}... (Captured {captured_in_category} URLs so far, No-image ASINs: {len(no_image_asins)})")
            # Periodically save mapping to avoid data loss on crash
            with open(URL_MAPPING_FILE, 'w') as f:
                json.dump(asin_to_image_url, f)
            # Save the no_image_asins set as a list for state persistence
            state_manager.state['no_image_asins'] = list(no_image_asins)
            state_manager.save_state()

        asin = row.get('parent_asin')

        # Skip if we already know this ASIN has no image
        if asin in no_image_asins:
            continue

        # Process only if it's a target ASIN and we haven't found its image yet
        if asin in target_asins and asin not in asin_to_image_url:
            images = row.get('images')
            img_url = None

            if not images:
                # No images field found, mark as no image
                no_image_asins.add(asin)
                continue

            # Robustly handle dirty / inconsistent data formats
            if isinstance(images, list) and len(images) > 0:
                first_img = images[0]
                if isinstance(first_img, dict):
                    img_url = first_img.get('hi_res') or first_img.get('large') or first_img.get('thumb')
                elif isinstance(first_img, str):
                    img_url = first_img
            elif isinstance(images, dict):
                hi_res = images.get('hi_res')
                large = images.get('large')
                if isinstance(hi_res, list) and hi_res:
                    img_url = hi_res[0]
                elif isinstance(large, list) and large:
                    img_url = large[0]
                elif isinstance(hi_res, str):
                    img_url = hi_res
                elif isinstance(large, str):
                    img_url = large

            if img_url and isinstance(img_url, str):
                asin_to_image_url[asin] = img_url
                captured_in_category += 1
            else:
                # If after parsing, still no valid URL, mark as no image
                no_image_asins.add(asin)

    # Save at the end of the category
    with open(URL_MAPPING_FILE, 'w') as f:
        json.dump(asin_to_image_url, f)

    state_manager.state['processed_meta_categories'][category] = True
    state_manager.state['no_image_asins'] = list(no_image_asins) # Save the set as a list
    state_manager.save_state()
    print(f"Completed metadata extraction for {category}. Total captured here: {captured_in_category}")

print(f"Total mapping size: {len(asin_to_image_url)}. Total ASINs identified with no image: {len(no_image_asins)}")

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loaded 4380665 ASIN image URLs from /content/drive/MyDrive/Amazon_Reviews_Pipeline/asin_to_image_url.json
Building ASIN to Image URL mapping from meta datasets...
Metadata for Clothing_Shoes_and_Jewelry already processed. Skipping.
Metadata for Sports_and_Outdoors already processed. Skipping.
Metadata for Beauty_and_Personal_Care already processed. Skipping.
Total mapping size: 4380665. Total ASINs identified with no image: 0


In [ ]:
current_state = state_manager.state

total_asins = len(current_state.get('unique_asins', []))
no_image_asins = len(current_state.get('no_image_asins', []))
target_asins = total_asins - no_image_asins
processed_index = current_state.get('last_processed_index', 0)
remaining_to_process = target_asins - processed_index

error_registry = current_state.get('error_registry', {})
dlq_count = 0
retrying_count = 0
missing_url_count = 0
invalid_url_count = 0
download_error_count = 0

for asin, info in error_registry.items():
    status = info.get('status')
    error_msg = info.get('error', '')

    if status == 'problematic':
        dlq_count += 1
    elif status == 'retrying':
        retrying_count += 1

    if "No image URL found" in error_msg:
        missing_url_count += 1
    elif "Invalid URL format" in error_msg:
        invalid_url_count += 1
    else:
        download_error_count += 1

print(f"--- Processing Status ---")
print(f"Total Unique ASINs: {total_asins:,}")
print(f"ASINs with known NO images (filtered): {no_image_asins:,}")
print(f"Target ASINs for processing: {target_asins:,}")
print(f"Processed Index: {processed_index:,}")
print(f"Remaining to Process (Queue): {remaining_to_process:,}")

print(f"\n--- Error Registry Breakdown ---")
print(f"Total items with issues: {len(error_registry):,}")
print(f"Dead Letter Queue (2+ strikes / 'problematic'): {dlq_count:,} (Will be skipped)")
print(f"Currently Retrying (1 strike): {retrying_count:,}")

print(f"\n--- Issue Types ---")
print(f"Missing URLs in metadata: {missing_url_count:,}")
print(f"Invalid URL formats: {invalid_url_count:,}")
print(f"Download/GPU Errors: {download_error_count:,}")

--- Processing Status ---
Total Unique ASINs: 4,381,511
ASINs with known NO images (filtered): 0
Target ASINs for processing: 4,381,511
Processed Index: 2,216,960
Remaining to Process (Queue): 2,164,551

--- Error Registry Breakdown ---
Total items with issues: 522
Dead Letter Queue (2+ strikes / 'problematic'): 0 (Will be skipped)
Currently Retrying (1 strike): 522

--- Issue Types ---
Missing URLs in metadata: 417
Invalid URL formats: 20
Download/GPU Errors: 85


# Step 3: GPU Processing

In [ ]:
import ast
import shutil
import concurrent.futures
import time
import math

# --- DISTRIBUTED WORKER CONFIGURATION ---
WORKER_ID = 1      # SET THIS TO 1, 2, or 3 IN YOUR DIFFERENT NOTEBOOKS
TOTAL_WORKERS = 3  # Total number of concurrent notebooks
# ----------------------------------------

# --- Step 3: Core Batch Processing & Local-First I/O (GPU BATCHED) ---
BATCH_SIZE = 2048
SYNC_EVERY_N_BATCHES = 10 # Save and sync every 10 batches (~5,000 records)
MAX_WORKERS = 128  # Number of parallel threads for downloading images

# Filter out ASINs that are known to have no images before starting the loop
initial_all_asins = state_manager.state['unique_asins']
no_image_asins_set = set(state_manager.state.get('no_image_asins', []))
all_asins_to_process = [asin for asin in initial_all_asins if asin not in no_image_asins_set]

# Calculate the remaining interval based on the master state
master_start_idx = state_manager.state['last_processed_index']
total_remaining = len(all_asins_to_process) - master_start_idx

# Split the remaining chunk
chunk_size = math.ceil(total_remaining / TOTAL_WORKERS)
worker_start_idx = master_start_idx + ((WORKER_ID - 1) * chunk_size)
worker_end_idx = min(worker_start_idx + chunk_size, len(all_asins_to_process))

# IMPORTANT: If WORKER_ID is the last worker, ensure it goes to the very end
if WORKER_ID == TOTAL_WORKERS:
    worker_end_idx = len(all_asins_to_process)

# Worker-specific state override to prevent Drive conflicts
WORKER_STATE_FILE = os.path.join(PROJECT_DIR, f'pipeline_state_worker_{WORKER_ID}.json')
# Copy main state to worker state if worker state doesn't exist to inherit historical errors
if not os.path.exists(WORKER_STATE_FILE):
    print(f"Creating isolated state file for Worker {WORKER_ID}...")
    shutil.copy2(STATE_FILE, WORKER_STATE_FILE)

worker_state_manager = PipelineState(WORKER_STATE_FILE)

# If the worker hasn't reached its assigned start block yet, advance it
# OR, for retrying all failed items, explicitly reset to worker_start_idx
# This ensures the worker re-scans its entire assigned chunk.
if worker_state_manager.state['last_processed_index'] < worker_start_idx:
    worker_state_manager.state['last_processed_index'] = worker_start_idx
    worker_state_manager.save_state()

current_idx = worker_state_manager.state['last_processed_index']

print(f"\n--- WORKER {WORKER_ID} / {TOTAL_WORKERS} ---")
print(f"Master start index: {master_start_idx:,}")
print(f"Worker assigned range: {worker_start_idx:,} to {worker_end_idx:,} (Total: {worker_end_idx - worker_start_idx:,})")
print(f"Worker resuming from: {current_idx:,}\n")

local_records = []
batch_count = 0

# Accumulators for sync-level reporting
sync_downloaded_images = 0
sync_valid_asins = 0
sync_download_errors = 0
sync_download_time = 0.0
sync_inference_time = 0.0
sync_url_parsing_time = 0.0
sync_clear_time = 0.0
sync_total_batch_time = 0.0

# Local directory for fast writes
LOCAL_EMBEDDINGS_DIR = os.path.join(LOCAL_TMP_DIR, 'embeddings')
os.makedirs(LOCAL_EMBEDDINGS_DIR, exist_ok=True)

def safe_download(url, asin):
    """Helper to download images safely in threads."""
    try:
        img = download_image(url, timeout=10)
        return asin, img, None
    except Exception as e:
        return asin, None, str(e)

# Iterate in chunks of BATCH_SIZE strictly within worker's bounds
for i in range(current_idx, worker_end_idx, BATCH_SIZE):
    batch_start_time = time.perf_counter()

    # Slice only up to the worker's max boundary
    end_of_chunk = min(i + BATCH_SIZE, worker_end_idx)
    batch_asins_chunk = all_asins_to_process[i : end_of_chunk]

    valid_batch_asins = []
    valid_batch_urls = []

    # 1. Filter and parse URLs for the current batch
    url_parsing_start_time = time.perf_counter()
    for asin in batch_asins_chunk:
        if worker_state_manager.is_problematic(asin):
            continue

        url = asin_to_image_url.get(asin)
        if not url:
            worker_state_manager.register_error(asin, "No image URL found in metadata.")
            continue

        if isinstance(url, str) and url.startswith("{") and url.endswith("}"):
            try:
                url_dict = ast.literal_eval(url)
                url = url_dict.get('hi_res') or url_dict.get('large') or url_dict.get('thumb')
            except:
                pass

        if not isinstance(url, str) or not url.startswith('http'):
            worker_state_manager.register_error(asin, f"Invalid URL format: {url}")
            continue

        valid_batch_asins.append(asin)
        valid_batch_urls.append(url)
    url_parsing_end_time = time.perf_counter()
    url_parsing_time = url_parsing_end_time - url_parsing_start_time

    downloaded_images = []
    download_errors_count = 0
    download_time = 0.0
    inference_time = 0.0
    clear_time = 0.0

    if valid_batch_asins:
        # 2. Download images concurrently
        download_start_time = time.perf_counter()
        successful_asins = []

        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(safe_download, u, a) for u, a in zip(valid_batch_urls, valid_batch_asins)]
            for future in concurrent.futures.as_completed(futures):
                asin, img, err = future.result()
                if img:
                    successful_asins.append(asin)
                    downloaded_images.append(img)
                else:
                    worker_state_manager.register_error(asin, err)
                    download_errors_count += 1
        download_end_time = time.perf_counter()
        download_time = download_end_time - download_start_time

        # 3. Batched GPU Inference
        inference_start_time = time.perf_counter()
        if downloaded_images:
            try:
                with torch.no_grad():
                    inputs = processor(images=downloaded_images, return_tensors="pt").to(device)
                    image_features = model.get_image_features(**inputs)

                    if hasattr(image_features, 'pooler_output'):
                        image_features = image_features.pooler_output
                    elif hasattr(image_features, 'image_embeds'):
                        image_features = image_features.image_embeds

                    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                    embeddings = image_features.cpu().numpy().tolist()

                for asin, emb in zip(successful_asins, embeddings):
                    local_records.append({'parent_asin': asin, 'embedding': emb})

                del inputs, image_features

            except Exception as e:
                for asin in successful_asins:
                    worker_state_manager.register_error(asin, f"Batch GPU error: {str(e)}")
        inference_end_time = time.perf_counter()
        inference_time = inference_end_time - inference_start_time

        # Force GC and empty VRAM cache
        gc_start_time = time.perf_counter()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc_end_time = time.perf_counter()
        clear_time = gc_end_time - gc_start_time

    # 4. Save and Sync to Drive
    sync_start_time = time.perf_counter()
    sync_time = 0.0
    is_sync_batch = (batch_count + 1) % SYNC_EVERY_N_BATCHES == 0 or end_of_chunk >= worker_end_idx

    if is_sync_batch:
        if local_records:
            df = pd.DataFrame(local_records)

            end_idx_for_file = end_of_chunk - 1

            # 1. Fast local write with Worker ID
            local_output_file = os.path.join(LOCAL_EMBEDDINGS_DIR, f'embeddings_w{WORKER_ID}_{end_idx_for_file}.parquet')
            df.to_parquet(local_output_file, engine='pyarrow', compression='snappy')

            # 2. Sync Parquet to Drive
            drive_output_file = os.path.join(FINAL_OUTPUT_DIR, f'embeddings_w{WORKER_ID}_{end_idx_for_file}.parquet')
            shutil.copy2(local_output_file, drive_output_file)

            local_records = []

        worker_state_manager.state['last_processed_index'] = end_of_chunk
        worker_state_manager.save_state()
        sync_end_time = time.perf_counter()
        sync_time = sync_end_time - sync_start_time

    batch_end_time = time.perf_counter()
    total_batch_time = batch_end_time - batch_start_time

    # Accumulate metrics
    sync_downloaded_images += len(downloaded_images)
    sync_valid_asins += len(valid_batch_asins)
    sync_download_errors += download_errors_count
    sync_download_time += download_time
    sync_inference_time += inference_time
    sync_url_parsing_time += url_parsing_time
    sync_clear_time += clear_time
    sync_total_batch_time += total_batch_time

    if is_sync_batch:
        print(f"[Worker {WORKER_ID}] Sync after Batch {batch_count}: Processed {worker_state_manager.state['last_processed_index']:,}/{worker_end_idx:,} | "
              f"URL Parse: {sync_url_parsing_time:.2f}s | "
              f"Downloaded: {sync_downloaded_images}/{sync_valid_asins} ({sync_download_errors} errs) in {sync_download_time:.2f}s | "
              f"GPU Inf: {sync_inference_time:.2f}s | "
              f"GC/VRAM: {sync_clear_time:.2f}s | "
              f"Sync Write: {sync_time:.2f}s | "
              f"Total Loop Time: {sync_total_batch_time:.2f}s")

        # Reset accumulators
        sync_downloaded_images = 0
        sync_valid_asins = 0
        sync_download_errors = 0
        sync_download_time = 0.0
        sync_inference_time = 0.0
        sync_url_parsing_time = 0.0
        sync_clear_time = 0.0
        sync_total_batch_time = 0.0

    for img in downloaded_images:
        img.close()

    batch_count += 1

print(f"Worker {WORKER_ID} core execution completed.")

Loaded state from /content/drive/MyDrive/Amazon_Reviews_Pipeline/pipeline_state_worker_1.json. Resuming from index 2938477

--- WORKER 1 / 3 ---
Master start index: 2,216,960
Worker assigned range: 2,216,960 to 2,938,477 (Total: 721,517)
Worker resuming from: 2,938,477

Worker 1 core execution completed.


### Final Sweep

Check out errored image downloads and retry.
- To try everything including dead letter items one moretime

In [ ]:
total_failures = len(state_manager.state.get('error_registry', {}))
print(f"Total number of failures recorded in error_registry: {total_failures:,}")

Total number of failures recorded in error_registry: 112,351


In [ ]:
worker_1_total_failures = len(worker_state_manager.state.get('error_registry', {}))
print(f"Total number of failures recorded in worker_1 error_registry: {worker_1_total_failures:,}")

Total number of failures recorded in worker_1 error_registry: 148,840


### Resetting Worker 1 Failure Registry for Retry

In [ ]:
# Reset the error registry for worker_1 to re-process all failed items
# Iterate over a copy of the keys to avoid issues when modifying the dictionary during iteration
for asin in list(worker_state_manager.state['error_registry'].keys()):
    # Set strikes to 0 and status to 'retrying' to ensure they are picked up again
    worker_state_manager.state['error_registry'][asin]['strikes'] = 0
    worker_state_manager.state['error_registry'][asin]['status'] = 'retrying'

# Save the updated state for worker_1
worker_state_manager.save_state()

print(f"Worker 1's error registry has been reset. Total items ready for retry: {len(worker_state_manager.state['error_registry']):,}")

# Verify the state for worker_1
worker_1_total_failures_after_reset = len(worker_state_manager.state.get('error_registry', {}))
print(f"Total number of failures recorded in worker_1 error_registry after reset: {worker_1_total_failures_after_reset:,}")

problematic_after_reset = 0
retrying_after_reset = 0
for asin, info in worker_state_manager.state['error_registry'].items():
    if info['status'] == 'problematic':
        problematic_after_reset += 1
    elif info['status'] == 'retrying':
        retrying_after_reset += 1

print(f"Problematic items after reset: {problematic_after_reset:,}")
print(f"Retrying items after reset: {retrying_after_reset:,}")

Worker 1's error registry has been reset. Total items ready for retry: 148,840
Total number of failures recorded in worker_1 error_registry after reset: 148,840
Problematic items after reset: 0
Retrying items after reset: 148,840


### Dedicated Retry Loop (Only processes failed items)

In [ ]:
import ast
import shutil
import concurrent.futures
import time
import gc
import pandas as pd
import torch
import os
from datetime import datetime

def t_print(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

# --- DEDICATED RETRY LOOP ---
# This bypasses the main index and ONLY targets items currently flagged as 'retrying'

BATCH_SIZE = 2048
MAX_WORKERS = 128

# 1. Isolate the items we specifically want to retry
retry_asins = [asin for asin, info in worker_state_manager.state['error_registry'].items() if info['status'] == 'retrying']
t_print(f"Found {len(retry_asins):,} specific items to retry.")

# Dedicated retry folder just to keep things clean (they sync to the main folder later)
RETRY_EMBEDDINGS_DIR = os.path.join(LOCAL_TMP_DIR, 'retry_embeddings')
os.makedirs(RETRY_EMBEDDINGS_DIR, exist_ok=True)

local_records = []
batch_count = 0

def safe_download(url, asin):
    try:
        img = download_image(url, timeout=20)
        return asin, img, None
    except Exception as e:
        return asin, None, str(e)

for i in range(0, len(retry_asins), BATCH_SIZE):
    batch_asins_chunk = retry_asins[i : i + BATCH_SIZE]

    valid_batch_asins = []
    valid_batch_urls = []

    # Parsing URLs
    for asin in batch_asins_chunk:
        url = asin_to_image_url.get(asin)

        if isinstance(url, str) and url.startswith("{") and url.endswith("}"):
            try:
                url_dict = ast.literal_eval(url)
                url = url_dict.get('hi_res') or url_dict.get('large') or url_dict.get('thumb')
            except:
                pass

        if not isinstance(url, str) or not url.startswith('http'):
            worker_state_manager.register_error(asin, f"Invalid URL format: {url}")
            continue

        valid_batch_asins.append(asin)
        valid_batch_urls.append(url)

    downloaded_images = []
    successful_asins = []

    # Downloading
    if valid_batch_asins:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(safe_download, u, a) for u, a in zip(valid_batch_urls, valid_batch_asins)]
            for future in concurrent.futures.as_completed(futures):
                asin, img, err = future.result()
                if img:
                    successful_asins.append(asin)
                    downloaded_images.append(img)
                else:
                    worker_state_manager.register_error(asin, err)

        # Inference
        if downloaded_images:
            try:
                with torch.no_grad():
                    inputs = processor(images=downloaded_images, return_tensors="pt").to(device)
                    image_features = model.get_image_features(**inputs)

                    if hasattr(image_features, 'pooler_output'):
                        image_features = image_features.pooler_output
                    elif hasattr(image_features, 'image_embeds'):
                        image_features = image_features.image_embeds

                    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                    embeddings = image_features.cpu().numpy().tolist()

                for asin, emb in zip(successful_asins, embeddings):
                    local_records.append({'parent_asin': asin, 'embedding': emb})

                    # CRITICAL: Remove from error registry if successful!
                    if asin in worker_state_manager.state['error_registry']:
                        del worker_state_manager.state['error_registry'][asin]

                del inputs, image_features
            except Exception as e:
                for asin in successful_asins:
                    worker_state_manager.register_error(asin, f"Batch GPU error: {str(e)}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Syncing to Drive
    is_sync_batch = (batch_count + 1) % 10 == 0 or (i + BATCH_SIZE) >= len(retry_asins)
    if is_sync_batch and local_records:
        df = pd.DataFrame(local_records)
        local_output_file = os.path.join(RETRY_EMBEDDINGS_DIR, f'retry_embeddings_w{WORKER_ID}_batch{batch_count}.parquet')
        df.to_parquet(local_output_file, engine='pyarrow', compression='snappy')

        drive_output_file = os.path.join(FINAL_OUTPUT_DIR, f'retry_embeddings_w{WORKER_ID}_batch{batch_count}.parquet')
        shutil.copy2(local_output_file, drive_output_file)

        local_records = []
        worker_state_manager.save_state()
        t_print(f"Processed retry batch {batch_count}. Remaining in Error Registry: {len(worker_state_manager.state['error_registry']):,}")

    for img in downloaded_images:
        img.close()

    batch_count += 1

t_print("Dedicated retry sweep completed!")


[2026-06-07 01:36:30] Found 7 specific items to retry.
[2026-06-07 01:36:32] Dedicated retry sweep completed!


### Resetting Main State Manager Failure Registry for Retry

In [ ]:
# Reset the error registry for the main state_manager
for asin in list(state_manager.state.get('error_registry', {}).keys()):
    state_manager.state['error_registry'][asin]['strikes'] = 0
    state_manager.state['error_registry'][asin]['status'] = 'retrying'

state_manager.save_state()

print(f"Main state manager's error registry has been reset. Total items ready for retry: {len(state_manager.state['error_registry']):,}")


Main state manager's error registry has been reset. Total items ready for retry: 112,351


### Dedicated Retry Loop (Main State Manager)

In [ ]:
import ast
import shutil
import concurrent.futures
import time
import gc
import pandas as pd
import torch
import os
from datetime import datetime

def t_print(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

BATCH_SIZE = 2048
MAX_WORKERS = 128
WORKER_ID = "MAIN" # Label for output files

retry_asins = [asin for asin, info in state_manager.state.get('error_registry', {}).items() if info['status'] == 'retrying']
t_print(f"Found {len(retry_asins):,} specific items to retry.")

RETRY_EMBEDDINGS_DIR = os.path.join(LOCAL_TMP_DIR, 'retry_embeddings_main')
os.makedirs(RETRY_EMBEDDINGS_DIR, exist_ok=True)

local_records = []
batch_count = 0

def safe_download(url, asin):
    try:
        img = download_image(url, timeout=20)
        return asin, img, None
    except Exception as e:
        return asin, None, str(e)

for i in range(0, len(retry_asins), BATCH_SIZE):
    batch_asins_chunk = retry_asins[i : i + BATCH_SIZE]

    valid_batch_asins = []
    valid_batch_urls = []

    for asin in batch_asins_chunk:
        url = asin_to_image_url.get(asin)

        if isinstance(url, str) and url.startswith("{") and url.endswith("}"):
            try:
                url_dict = ast.literal_eval(url)
                url = url_dict.get('hi_res') or url_dict.get('large') or url_dict.get('thumb')
            except:
                pass

        if not isinstance(url, str) or not url.startswith('http'):
            state_manager.register_error(asin, f"Invalid URL format: {url}")
            continue

        valid_batch_asins.append(asin)
        valid_batch_urls.append(url)

    downloaded_images = []
    successful_asins = []

    if valid_batch_asins:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(safe_download, u, a) for u, a in zip(valid_batch_urls, valid_batch_asins)]
            for future in concurrent.futures.as_completed(futures):
                asin, img, err = future.result()
                if img:
                    successful_asins.append(asin)
                    downloaded_images.append(img)
                else:
                    state_manager.register_error(asin, err)

        if downloaded_images:
            try:
                with torch.no_grad():
                    inputs = processor(images=downloaded_images, return_tensors="pt").to(device)
                    image_features = model.get_image_features(**inputs)

                    if hasattr(image_features, 'pooler_output'):
                        image_features = image_features.pooler_output
                    elif hasattr(image_features, 'image_embeds'):
                        image_features = image_features.image_embeds

                    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                    embeddings = image_features.cpu().numpy().tolist()

                for asin, emb in zip(successful_asins, embeddings):
                    local_records.append({'parent_asin': asin, 'embedding': emb})

                    if asin in state_manager.state['error_registry']:
                        del state_manager.state['error_registry'][asin]

                del inputs, image_features
            except Exception as e:
                for asin in successful_asins:
                    state_manager.register_error(asin, f"Batch GPU error: {str(e)}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    is_sync_batch = (batch_count + 1) % 10 == 0 or (i + BATCH_SIZE) >= len(retry_asins)
    if is_sync_batch and local_records:
        df = pd.DataFrame(local_records)
        local_output_file = os.path.join(RETRY_EMBEDDINGS_DIR, f'retry_embeddings_{WORKER_ID}_batch{batch_count}.parquet')
        df.to_parquet(local_output_file, engine='pyarrow', compression='snappy')

        drive_output_file = os.path.join(FINAL_OUTPUT_DIR, f'retry_embeddings_{WORKER_ID}_batch{batch_count}.parquet')
        shutil.copy2(local_output_file, drive_output_file)

        local_records = []
        state_manager.save_state()
        t_print(f"Processed retry batch {batch_count}. Remaining in Error Registry: {len(state_manager.state['error_registry']):,}")

    for img in downloaded_images:
        img.close()

    batch_count += 1

t_print("Dedicated retry sweep completed for main state manager!")


[2026-06-07 01:39:50] Found 112,351 specific items to retry.
[2026-06-07 01:45:22] Processed retry batch 9. Remaining in Error Registry: 91,963
[2026-06-07 01:50:54] Processed retry batch 19. Remaining in Error Registry: 71,577
[2026-06-07 01:56:12] Processed retry batch 29. Remaining in Error Registry: 51,190
[2026-06-07 02:01:44] Processed retry batch 39. Remaining in Error Registry: 30,814
[2026-06-07 02:07:10] Processed retry batch 49. Remaining in Error Registry: 10,429
[2026-06-07 02:09:46] Processed retry batch 54. Remaining in Error Registry: 522
[2026-06-07 02:09:46] Dedicated retry sweep completed for main state manager!


In [ ]:
# Extract a sample of malformed URLs from the error registry
invalid_url_samples = []
for asin, info in worker_state_manager.state['error_registry'].items():
    if "Invalid URL format" in info.get('error', ''):
        invalid_url_samples.append((asin, info.get('error')))
        if len(invalid_url_samples) >= 5:
            break

print("--- Sample of Malformed URLs ---")
for i, (asin, error_msg) in enumerate(invalid_url_samples, 1):
    print(f"{i}. ASIN: {asin}")
    print(f"   Details: {error_msg}\n")


--- Sample of Malformed URLs ---
1. ASIN: B07QYNS4DR
   Details: Invalid URL format: {'thumb': 'https://m.media-amazon.com/images/I/4133UMYehuL._AC_SR38,50_.jpg', 'large': 'https://m.media-amazon.com/images/I/4133UMYehuL._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/91bEX203SYL._AC_UL1500_.jpg'}

2. ASIN: B07Y9NFHLN
   Details: Invalid URL format: {'thumb': 'https://m.media-amazon.com/images/I/41OXF6R2vyL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41OXF6R2vyL._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/61vo9CUBh0L._AC_UL1000_.jpg'}

3. ASIN: B09C39ZVBG
   Details: Invalid URL format: {'thumb': 'https://m.media-amazon.com/images/I/51MM2ZdZHzL._AC_SR38,50_.jpg', 'large': 'https://m.media-amazon.com/images/I/51MM2ZdZHzL._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/81CRz7H7m9L._AC_UL1500_.jpg'}

4. ASIN: B07BNYZP7B
   Details: Invalid URL format: {'thumb': 'https://m.media-ama

In [ ]:
import ast
import json

fixed_count = 0
asins_to_reset = []

# 1. Fix the in-memory URL mapping dictionary
print("Cleaning up malformed URLs...")
for asin, url in asin_to_image_url.items():
    clean_url = None

    # Check if the URL is actually a Python dictionary
    if isinstance(url, dict):
        clean_url = url.get('hi_res') or url.get('large') or url.get('thumb')

    # Fallback just in case some are actually stringified dictionaries
    elif isinstance(url, str) and url.startswith("{") and url.endswith("}"):
        try:
            url_dict = ast.literal_eval(url)
            clean_url = url_dict.get('hi_res') or url_dict.get('large') or url_dict.get('thumb')
        except Exception:
            pass

    if clean_url and clean_url != url:
        asin_to_image_url[asin] = clean_url
        fixed_count += 1
        asins_to_reset.append(asin)

print(f"Fixed {fixed_count:,} malformed URLs in the mapping dictionary.")

# 2. Save the fixed mapping back to Drive to persist the corrections
with open(URL_MAPPING_FILE, 'w') as f:
    json.dump(asin_to_image_url, f)
print("Saved updated URL mapping to Drive.")

# 3. Reset the error registry for these specific ASINs so they can be retried
reset_count = 0
for asin in asins_to_reset:
    if asin in worker_state_manager.state['error_registry']:
        worker_state_manager.state['error_registry'][asin]['strikes'] = 0
        worker_state_manager.state['error_registry'][asin]['status'] = 'retrying'
        reset_count += 1

worker_state_manager.save_state()
print(f"Reset {reset_count:,} fixed items in Worker 1's error registry back to 'retrying' status.")
print("\nYou can now re-run the 'Dedicated Retry Loop' cell above to process these items!")


Cleaning up malformed URLs...
Fixed 218,155 malformed URLs in the mapping dictionary.
Saved updated URL mapping to Drive.
Reset 146,757 fixed items in Worker 1's error registry back to 'retrying' status.

You can now re-run the 'Dedicated Retry Loop' cell above to process these items!


### Querying data that is linked

In [ ]:
import pandas as pd
import glob
import json
import requests
import os

print("--- Demonstrating Metadata Linkage ---")

# 1. Find one of your generated parquet files
parquet_files = glob.glob(os.path.join(FINAL_OUTPUT_DIR, '*.parquet'))

if not parquet_files:
    print("No parquet files found yet! Run the pipeline to generate some first.")
else:
    test_file = parquet_files[0]
    print(f"1. Loading generated embeddings from: {test_file}")
    df_emb = pd.read_parquet(test_file)

    # Take 3 ASINs to test
    sample_asins = set(df_emb['parent_asin'].head(3))
    print(f"\nSample ASINs to lookup: {sample_asins}")

    # 2. Fetch corresponding metadata from the HuggingFace source
    # (In a real scenario, you might download this file fully to disk first)
    category = CATEGORIES[0] # Let's check the first category
    url = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_{category}.jsonl"

    print(f"\n2. Scanning metadata stream for these ASINs...")
    response = requests.get(url, stream=True)

    metadata_records = []
    lines_checked = 0

    for line in response.iter_lines():
        if not line: continue
        lines_checked += 1

        row = json.loads(line)
        if row.get('parent_asin') in sample_asins:
            metadata_records.append({
                'parent_asin': row.get('parent_asin'),
                'title': row.get('title'),
                'main_category': row.get('main_category'),
                'price': row.get('price')
            })

        # Stop once we found our samples, or after checking 200,000 rows to prevent hanging
        if len(metadata_records) == len(sample_asins) or lines_checked > 200000:
            break

    # 3. Join the data!
    if metadata_records:
        df_meta = pd.DataFrame(metadata_records)

        print("\n3. Joining Embeddings with Original Metadata on 'parent_asin':")
        df_joined = pd.merge(df_emb, df_meta, on='parent_asin', how='inner')

        # Display the result (dropping the massive 512-dim embedding column just so it's readable here)
        display(df_joined.drop(columns=['embedding']))
        print(f"\n(Embedding vector of length {len(df_joined.iloc[0]['embedding'])} is present but hidden for display)")
    else:
        print("\nCould not find these specific ASINs in the first 200,000 rows of the meta file. (They might be deeper in the file or in another category).")

### Resetting and Retrying Worker 3

In [ ]:
worker_3_state_file = os.path.join(PROJECT_DIR, 'pipeline_state_worker_3.json')
worker_3_state_manager = PipelineState(worker_3_state_file)

# Reset the error registry for worker_3
for asin in list(worker_3_state_manager.state.get('error_registry', {}).keys()):
    worker_3_state_manager.state['error_registry'][asin]['strikes'] = 0
    worker_3_state_manager.state['error_registry'][asin]['status'] = 'retrying'

worker_3_state_manager.save_state()
print(f"Worker 3's error registry has been reset. Total items ready for retry: {len(worker_3_state_manager.state.get('error_registry', {})):,}")

Loaded state from /content/drive/MyDrive/Amazon_Reviews_Pipeline/pipeline_state_worker_3.json. Resuming from index 4381511
Worker 3's error registry has been reset. Total items ready for retry: 67,293


In [ ]:
from datetime import datetime

def t_print(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

WORKER_ID = 3

retry_asins = [asin for asin, info in worker_3_state_manager.state.get('error_registry', {}).items() if info['status'] == 'retrying']
t_print(f"Found {len(retry_asins):,} specific items to retry for Worker {WORKER_ID}.")

RETRY_EMBEDDINGS_DIR = os.path.join(LOCAL_TMP_DIR, f'retry_embeddings_w{WORKER_ID}')
os.makedirs(RETRY_EMBEDDINGS_DIR, exist_ok=True)

local_records = []
batch_count = 0

for i in range(0, len(retry_asins), BATCH_SIZE):
    batch_asins_chunk = retry_asins[i : i + BATCH_SIZE]

    valid_batch_asins = []
    valid_batch_urls = []

    for asin in batch_asins_chunk:
        url = asin_to_image_url.get(asin)

        if isinstance(url, str) and url.startswith("{") and url.endswith("}"):
            try:
                url_dict = ast.literal_eval(url)
                url = url_dict.get('hi_res') or url_dict.get('large') or url_dict.get('thumb')
            except:
                pass

        if not isinstance(url, str) or not url.startswith('http'):
            worker_3_state_manager.register_error(asin, f"Invalid URL format: {url}")
            continue

        valid_batch_asins.append(asin)
        valid_batch_urls.append(url)

    downloaded_images = []
    successful_asins = []

    if valid_batch_asins:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(safe_download, u, a) for u, a in zip(valid_batch_urls, valid_batch_asins)]
            for future in concurrent.futures.as_completed(futures):
                asin, img, err = future.result()
                if img:
                    successful_asins.append(asin)
                    downloaded_images.append(img)
                else:
                    worker_3_state_manager.register_error(asin, err)

        if downloaded_images:
            try:
                with torch.no_grad():
                    inputs = processor(images=downloaded_images, return_tensors="pt").to(device)
                    image_features = model.get_image_features(**inputs)

                    if hasattr(image_features, 'pooler_output'):
                        image_features = image_features.pooler_output
                    elif hasattr(image_features, 'image_embeds'):
                        image_features = image_features.image_embeds

                    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                    embeddings = image_features.cpu().numpy().tolist()

                for asin, emb in zip(successful_asins, embeddings):
                    local_records.append({'parent_asin': asin, 'embedding': emb})

                    if asin in worker_3_state_manager.state['error_registry']:
                        del worker_3_state_manager.state['error_registry'][asin]

                del inputs, image_features
            except Exception as e:
                for asin in successful_asins:
                    worker_3_state_manager.register_error(asin, f"Batch GPU error: {str(e)}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    is_sync_batch = (batch_count + 1) % 10 == 0 or (i + BATCH_SIZE) >= len(retry_asins)
    if is_sync_batch and local_records:
        df = pd.DataFrame(local_records)
        local_output_file = os.path.join(RETRY_EMBEDDINGS_DIR, f'retry_embeddings_w{WORKER_ID}_batch{batch_count}.parquet')
        df.to_parquet(local_output_file, engine='pyarrow', compression='snappy')

        drive_output_file = os.path.join(FINAL_OUTPUT_DIR, f'retry_embeddings_w{WORKER_ID}_batch{batch_count}.parquet')
        shutil.copy2(local_output_file, drive_output_file)

        local_records = []
        worker_3_state_manager.save_state()
        t_print(f"Processed retry batch {batch_count}. Remaining in Error Registry: {len(worker_3_state_manager.state['error_registry']):,}")

    for img in downloaded_images:
        img.close()

    batch_count += 1

t_print(f"Dedicated retry sweep completed for Worker {WORKER_ID}!")

[2026-06-07 07:16:30] Found 67,293 specific items to retry for Worker 3.
[2026-06-07 07:22:01] Processed retry batch 9. Remaining in Error Registry: 47,280
[2026-06-07 07:27:23] Processed retry batch 19. Remaining in Error Registry: 26,893
[2026-06-07 07:32:56] Processed retry batch 29. Remaining in Error Registry: 6,505
[2026-06-07 07:34:37] Processed retry batch 32. Remaining in Error Registry: 680
[2026-06-07 07:34:37] Dedicated retry sweep completed for Worker 3!
